In [1]:
import pandas as pd
from scipy.spatial.distance import pdist, cdist
import numpy as np
import math
import datetime as dt
from datetime import timedelta
import itertools
from collections import defaultdict
import geopandas as gpd
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../tests")))

In [2]:
import filters
import daphmeIO as loader
import constants as constants
import stop_detection_modified as SD

## Load data/sample4

```sample4``` has columns representing user ID, longitude, latitude, and datetime.

In [3]:
path = '../data/sample4/'

traj_cols = {'user_id':'user_id',
             'latitude':'dev_lat',
             'longitude':'dev_lon',
             'datetime':'local_datetime'}

sample4 = loader.from_file(path, traj_cols=traj_cols, format='csv')

In [4]:
sample4.head()

,user_id,dev_lat,dev_lon,local_datetime
0,wizardly_joliot,38.321711,-36.667334,2024-01-01 14:29:00
1,wizardly_joliot,38.321676,-36.667365,2024-01-01 14:35:00
2,wonderful_swirles,38.321017,-36.667869,2024-01-01 15:06:00
3,youthful_galileo,38.321625,-36.666612,2024-01-01 08:47:00
4,youthful_galileo,38.321681,-36.666841,2024-01-01 09:59:00


To showcase how the algorithms work in a time efficient manner, we can first filter to one user only.

In [5]:
user_sample4 = sample4[sample4['user_id'] == "angry_spence"]

The data could also come with different formats for spatial and temporal variables:

In [6]:
# Longitude, Latitude, timestamp (unix time)
traj_cols_lon_lat_timestamp = {'user_id':'user_id',
                               'latitude':'dev_lat',
                               'longitude':'dev_lon',
                               'timestamp':'unix_time'}

user_sample4_lon_lat_timestamp = user_sample4.copy()
user_sample4_lon_lat_timestamp['unix_time'] = user_sample4_lon_lat_timestamp['local_datetime'].astype('int64') // 10**9

# x, y, datetime
traj_cols_x_y_datetime = {'user_id':'user_id',
                          'x':'x',
                          'y':'y',
                          'datetime':'local_datetime'}

user_sample4_x_y_datetime = user_sample4.copy()
user_sample4_x_y_datetime = filters.to_projection(user_sample4_x_y_datetime, longitude='dev_lon', latitude='dev_lat')

# x, y, timestamp (unix time)
traj_cols_x_y_timestamp = {'user_id':'user_id',
                          'x':'x',
                          'y':'y',
                          'timestamp':'unix_time'}

user_sample4_x_y_timestamp = user_sample4.copy()
user_sample4_x_y_timestamp = filters.to_projection(user_sample4_x_y_timestamp, longitude='dev_lon', latitude='dev_lat')
user_sample4_x_y_timestamp['unix_time'] = user_sample4_x_y_timestamp['local_datetime'].astype('int64') // 10**9

## Stop Detection Algorithms

### Lachesis

We need to pass some reasonable parameters for:
* ```dur_min```: Minimum duration for a stay in minutes.
* ```dt_max```: Maximum duration permitted between consecutive pings in a stay in minutes (dt_max should be greater than dur_min).
* ```delta_roam```: Maximum roaming distance for a stay in meters.

In [7]:
DUR_MIN = 60
DT_MAX = 120
DELTA_ROAM = 50

Minimum stop table output columns:

In [8]:
SD.lachesis(user_sample4, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols, complete_output=False)

,start_time,duration,x,y
0,2024-01-01 15:04:00,0 days 01:12:00,-36.66647,38.320064
1,2024-01-01 18:17:00,0 days 02:41:00,-36.667398,38.320415
2,2024-01-01 21:15:00,0 days 03:55:00,-36.667525,38.321253
3,2024-01-02 05:54:00,0 days 02:07:00,-36.667489,38.321273
4,2024-01-02 08:40:00,0 days 03:19:00,-36.667721,38.320554
5,2024-01-03 09:26:00,0 days 01:01:00,-36.667738,38.320675
6,2024-01-03 10:39:00,0 days 01:17:00,-36.667853,38.321051
7,2024-01-03 13:01:00,0 days 04:36:00,-36.667299,38.320045
8,2024-01-03 18:06:00,0 days 07:10:00,-36.66676,38.32098
9,2024-01-04 01:19:00,0 days 01:13:00,-36.667507,38.321394


All stop table output columns:

In [9]:
SD.lachesis(user_sample4, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols, complete_output=True)

,start_time,end_time,x,y,diameter,n_pings,duration
0,2024-01-01 15:04:00,2024-01-01 16:16:00,-36.66647,38.320064,14.691535,14,0 days 01:12:00
1,2024-01-01 18:17:00,2024-01-01 20:58:00,-36.667398,38.320415,12.974932,19,0 days 02:41:00
2,2024-01-01 21:15:00,2024-01-02 01:10:00,-36.667525,38.321253,13.946715,32,0 days 03:55:00
3,2024-01-02 05:54:00,2024-01-02 08:01:00,-36.667489,38.321273,31.46013,17,0 days 02:07:00
4,2024-01-02 08:40:00,2024-01-02 11:59:00,-36.667721,38.320554,20.736412,39,0 days 03:19:00
5,2024-01-03 09:26:00,2024-01-03 10:27:00,-36.667738,38.320675,49.433588,12,0 days 01:01:00
6,2024-01-03 10:39:00,2024-01-03 11:56:00,-36.667853,38.321051,24.082471,8,0 days 01:17:00
7,2024-01-03 13:01:00,2024-01-03 17:37:00,-36.667299,38.320045,47.749101,44,0 days 04:36:00
8,2024-01-03 18:06:00,2024-01-04 01:16:00,-36.66676,38.32098,33.545006,48,0 days 07:10:00
9,2024-01-04 01:19:00,2024-01-04 02:32:00,-36.667507,38.321394,16.765689,15,0 days 01:13:00


Lachesis with longitude, latitude, and timestamp (unix time):

In [10]:
SD.lachesis(user_sample4_lon_lat_timestamp, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_lon_lat_timestamp, complete_output=False)

,start_time,duration,x,y
0,1.704121e+09,4320.0,-36.666470,38.320064
1,1.704133e+09,9660.0,-36.667398,38.320415
2,1.704144e+09,14100.0,-36.667525,38.321253
3,1.704175e+09,7620.0,-36.667489,38.321273
4,1.704185e+09,11940.0,-36.667721,38.320554
5,1.704274e+09,3660.0,-36.667738,38.320675
6,1.704278e+09,4620.0,-36.667853,38.321051
7,1.704287e+09,16560.0,-36.667299,38.320045
8,1.704305e+09,25800.0,-36.666760,38.320980
9,1.704331e+09,4380.0,-36.667507,38.321394


Lachesis with x, y, and timestamp (unix time):

In [11]:
SD.lachesis(user_sample4_x_y_timestamp, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_x_y_timestamp, complete_output=False)

,start_time,duration,x,y
0,1.704121e+09,4320.0,-4.081693e+06,4.624739e+06
1,1.704133e+09,9660.0,-4.081796e+06,4.624789e+06
2,1.704144e+09,14100.0,-4.081810e+06,4.624908e+06
3,1.704175e+09,7620.0,-4.081806e+06,4.624911e+06
4,1.704185e+09,11940.0,-4.081832e+06,4.624809e+06
5,1.704276e+09,6660.0,-4.081845e+06,4.624889e+06
6,1.704291e+09,12960.0,-4.081785e+06,4.624737e+06
7,1.704305e+09,25800.0,-4.081725e+06,4.624869e+06
8,1.704331e+09,4380.0,-4.081808e+06,4.624928e+06
9,1.704344e+09,10980.0,-4.081808e+06,4.624927e+06


Lachesis with x, y, and datetime:

In [12]:
SD.lachesis(user_sample4_x_y_datetime, DUR_MIN, DT_MAX, DELTA_ROAM, traj_cols=traj_cols_x_y_datetime, complete_output=False)

,start_time,duration,x,y
0,2024-01-01 15:04:00,0 days 01:12:00,-4081692.766601,4624739.339131
1,2024-01-01 18:17:00,0 days 02:41:00,-4081796.107506,4624789.158858
2,2024-01-01 21:15:00,0 days 03:55:00,-4081810.192229,4624907.987139
3,2024-01-02 05:54:00,0 days 02:07:00,-4081806.158897,4624910.902451
4,2024-01-02 08:40:00,0 days 03:19:00,-4081832.070485,4624808.802068
5,2024-01-03 10:05:00,0 days 01:51:00,-4081844.834337,4624889.002115
6,2024-01-03 14:03:00,0 days 03:36:00,-4081785.058141,4624736.681099
7,2024-01-03 18:06:00,0 days 07:10:00,-4081725.097327,4624869.336981
8,2024-01-04 01:19:00,0 days 01:13:00,-4081808.204314,4624927.97637
9,2024-01-04 04:54:00,0 days 03:03:00,-4081807.596107,4624927.360419


### Temporal DBSCAN

We need to pass some reasonable parameters for:
* ```time_thresh```: Time threshold in minutes for identifying neighbors.
* ```dist_thresh```: Distance threshold in meters for identifying neighbors.
* ```min_pts```: Minimum number of points required to form a dense region (core point).

In [13]:
TIME_THRESH = 100
DIST_THRESH = 40
MIN_PTS = 10

We can get the final cluster and core labels for each of the data points:

In [14]:
SD._temporal_dbscan_labels(user_sample4, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols)

,cluster,core
2024-01-01 10:21:00,-1,-1
2024-01-01 10:27:00,-1,-1
2024-01-01 10:29:00,-1,-1
2024-01-01 10:39:00,-1,-1
2024-01-01 10:42:00,-1,-1
...,...,...
2024-01-15 07:23:00,1,1
2024-01-15 07:29:00,1,1
2024-01-15 07:33:00,1,1
2024-01-15 07:39:00,1,1


Or we can also get a stop table:

In [15]:
SD.temporal_dbscan(user_sample4, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols)

,start_time,duration,x,y
cluster,,,,
0,2024-01-04 16:02:00,0 days 00:43:00,-0.639973,0.668841
1,2024-01-14 19:45:00,0 days 11:54:00,-0.639968,0.668818
2,2024-01-14 14:10:00,0 days 01:29:00,-0.639963,0.668810
3,2024-01-14 01:45:00,0 days 06:07:00,-0.639956,0.668827
4,2024-01-13 13:53:00,0 days 02:52:00,-0.639952,0.668824
5,2024-01-13 09:29:00,0 days 02:31:00,-0.639974,0.668821
6,2024-01-12 19:01:00,0 days 10:08:00,-0.639956,0.668827
7,2024-01-12 14:49:00,0 days 02:33:00,-0.639974,0.668820
8,2024-01-12 08:02:00,0 days 02:38:00,-0.639956,0.668839


Temporal DBSCAN with longitude, latitude, and timestamp (unix time):

In [16]:
SD.temporal_dbscan(user_sample4_lon_lat_timestamp, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_lon_lat_timestamp)

,start_time,duration,x,y
cluster,,,,
0,1.704384e+09,2580.0,-0.639973,0.668841
1,1.705262e+09,42840.0,-0.639968,0.668818
2,1.705241e+09,5340.0,-0.639963,0.668810
3,1.705197e+09,22020.0,-0.639956,0.668827
4,1.705154e+09,10320.0,-0.639952,0.668824
5,1.705138e+09,9060.0,-0.639974,0.668821
6,1.705086e+09,36480.0,-0.639956,0.668827
7,1.705071e+09,9180.0,-0.639974,0.668820
8,1.705047e+09,9480.0,-0.639956,0.668839


Temporal DBSCAN with x, y, and timestamp (unix time):

In [17]:
SD.temporal_dbscan(user_sample4_x_y_timestamp, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_x_y_timestamp)

,start_time,duration,x,y
cluster,,,,
0,1.704384e+09,2580.0,-4.081836e+06,4.624984e+06
1,1.705262e+09,42840.0,-4.081802e+06,4.624791e+06
2,1.705241e+09,5340.0,-4.081771e+06,4.624728e+06
3,1.705197e+09,22020.0,-4.081725e+06,4.624865e+06
4,1.705158e+09,5940.0,-4.081703e+06,4.624845e+06
5,1.705138e+09,9060.0,-4.081840e+06,4.624816e+06
6,1.705086e+09,36480.0,-4.081724e+06,4.624870e+06
7,1.705071e+09,9180.0,-4.081839e+06,4.624814e+06
8,1.705047e+09,9480.0,-4.081729e+06,4.624965e+06


Temporal DBSCAN with x, y, and datetime:

In [18]:
SD.temporal_dbscan(user_sample4_x_y_datetime, TIME_THRESH, DIST_THRESH, MIN_PTS, complete_output=False, traj_cols=traj_cols_x_y_datetime)

,start_time,duration,x,y
cluster,,,,
0,2024-01-04 16:02:00,0 days 00:43:00,-4.081836e+06,4.624984e+06
1,2024-01-14 19:45:00,0 days 11:54:00,-4.081802e+06,4.624791e+06
2,2024-01-14 14:10:00,0 days 01:29:00,-4.081771e+06,4.624728e+06
3,2024-01-14 01:45:00,0 days 06:07:00,-4.081725e+06,4.624865e+06
4,2024-01-13 15:06:00,0 days 01:39:00,-4.081703e+06,4.624845e+06
5,2024-01-13 09:29:00,0 days 02:31:00,-4.081840e+06,4.624816e+06
6,2024-01-12 19:01:00,0 days 10:08:00,-4.081724e+06,4.624870e+06
7,2024-01-12 14:49:00,0 days 02:33:00,-4.081839e+06,4.624814e+06
8,2024-01-12 08:02:00,0 days 02:38:00,-4.081729e+06,4.624965e+06
